# pipe_nuevo — 01 Preprocesamiento: zero-fill al estilo del profe

Densifica `sell-in` con ceros usando la misma lógica que
`src/workflow/z601_workflow_ceros.ipynb`: el producto cartesiano
cliente × producto × período, acotado por la **vida del producto**
(`periodo_min`/`periodo_max` de ESE producto, sobre todos los clientes) y la
**fecha de alta del cliente** (`periodo_min` de ESE cliente, sobre todos los
productos) — no por la vida propia de cada par cliente-producto, que es lo que
hace `pipe_2/01_Preproceso_y_FE.ipynb`.

Con esto quedan filas explícitas en cero también para pares cliente-producto
que **nunca se transaron** pero pudieron haberlo hecho (cliente y producto
coexistiendo). Dataset de entrenamiento más grande, con señal explícita de
"este cliente no compra este producto".

Corre en DuckDB (como el notebook original) para el cross join pesado —
miles de clientes × cientos de productos × ~36 meses — y deja el resultado en
parquet para que `02_FE.ipynb` lo lea sin tener que rehacerlo.


In [ ]:
import os, json, time
from pathlib import Path

import duckdb


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_PRE = BUCKET / "datasets_pre"     # de aca lee 02_FE de pipe_nuevo
DIR_PRE.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"salida : {DIR_PRE}")


In [ ]:
PARAM = {
    # 'pc' -> cliente-producto: cartesiano cliente x producto x periodo, estilo z601.
    # 'p'  -> producto solo: no hay dimension cliente, solo se rellenan los huecos
    #         dentro de la vida del producto (no hace falta cruzar con clientes).
    'granularidad': 'pc',

    # Solo los productos que se entregan, ANTES de armar el cartesiano: sin este
    # filtro la matriz cliente x producto explota (todos los productos x todos los
    # clientes x todos los periodos).
    'solo_productos_target': True,

    'semilla': 102191,
}

G = PARAM['granularidad']
ES_PC = G == 'pc'

_grp = 'grpClienteProducto' if ES_PC else 'grpProducto'
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
NOMBRE = f"sellin_zeroes_{_grp}{_tgt}.parquet"

print(f"granularidad: {G}   (ES_PC={ES_PC})")
print(f"salida: {NOMBRE}")


In [ ]:
t0 = time.time()
con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT customer_id, product_id, periodo, plan_precios_cuidados,
           cust_request_qty, cust_request_tn, tn
    FROM read_csv_auto('{DIR_RAW / "sell-in.txt.gz"}')
    ORDER BY customer_id, product_id, periodo
""")

if PARAM['solo_productos_target']:
    con.execute(f"""
        CREATE OR REPLACE TABLE tb_target AS
        SELECT DISTINCT product_id
        FROM read_csv_auto('{DIR_RAW / "product_id_apredecir201912.txt"}')
    """)
    antes = con.sql("SELECT COUNT(DISTINCT product_id) FROM tb_sellin").fetchone()[0]
    con.execute("""
        CREATE OR REPLACE TABLE tb_sellin AS
        SELECT s.* FROM tb_sellin s
        WHERE s.product_id IN (SELECT product_id FROM tb_target)
    """)
    despues = con.sql("SELECT COUNT(DISTINCT product_id) FROM tb_sellin").fetchone()[0]
    print(f"solo productos target: {antes} -> {despues} productos")

n_sellin = con.sql("SELECT COUNT(*) FROM tb_sellin").fetchone()[0]
print(f"tb_sellin: {n_sellin:,} filas")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()

# la fecha de nacimiento y muerte de los productos
con.execute("""
    CREATE OR REPLACE TABLE tb_productos_fechas AS
    SELECT product_id, MIN(periodo) AS periodo_min, MAX(periodo) AS periodo_max
    FROM tb_sellin GROUP BY product_id
""")

# la fecha de alta (primera compra) de cada cliente
con.execute("""
    CREATE OR REPLACE TABLE tb_clientes_fechas AS
    SELECT customer_id, MIN(periodo) AS periodo_min
    FROM tb_sellin GROUP BY customer_id
""")

con.execute("""
    CREATE OR REPLACE TABLE tb_periodos AS
    SELECT DISTINCT periodo FROM tb_sellin ORDER BY 1
""")

con.execute("""
    CREATE OR REPLACE TABLE tb_precios_cuidados AS
    SELECT product_id, MIN(periodo) AS periodo_min, MAX(periodo) AS periodo_max
    FROM tb_sellin WHERE plan_precios_cuidados = 1
    GROUP BY product_id ORDER BY 1
""")

print(f"periodos : {con.sql('SELECT COUNT(*) FROM tb_periodos').fetchone()[0]}")
print(f"productos: {con.sql('SELECT COUNT(*) FROM tb_productos_fechas').fetchone()[0]}")
print(f"clientes : {con.sql('SELECT COUNT(*) FROM tb_clientes_fechas').fetchone()[0]}")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()

if ES_PC:
    # Cartesiano cliente x producto x periodo, acotado por la vida del PRODUCTO y
    # la fecha de alta del CLIENTE (no la vida del PAR): asi quedan en cero tambien
    # los pares que nunca se transaron pero pudieron haberlo hecho.
    con.execute("""
        CREATE OR REPLACE TABLE tb_zeroes AS
        SELECT cf.customer_id, pf.product_id, per.periodo,
               CAST(0 AS INT) AS plan_precios_cuidados,
               CAST(0 AS INT) AS cust_request_qty,
               0.0 AS cust_request_tn,
               0.0 AS tn
        FROM tb_productos_fechas pf, tb_clientes_fechas cf, tb_periodos per
        WHERE NOT EXISTS (
                SELECT 1 FROM tb_sellin si
                WHERE si.periodo = per.periodo
                  AND si.customer_id = cf.customer_id
                  AND si.product_id = pf.product_id
              )
          AND per.periodo BETWEEN pf.periodo_min AND pf.periodo_max
          AND per.periodo >= cf.periodo_min
        ORDER BY 1, 2, 3
    """)
else:
    # Sin dimension cliente: solo completar los periodos sin venta dentro de la
    # vida del producto (equivalente a 'densificar':'vida' a nivel producto).
    con.execute("""
        CREATE OR REPLACE TABLE tb_zeroes AS
        SELECT CAST(NULL AS BIGINT) AS customer_id, pf.product_id, per.periodo,
               CAST(0 AS INT) AS plan_precios_cuidados,
               CAST(0 AS INT) AS cust_request_qty,
               0.0 AS cust_request_tn,
               0.0 AS tn
        FROM tb_productos_fechas pf, tb_periodos per
        WHERE NOT EXISTS (
                SELECT 1 FROM tb_sellin si
                WHERE si.periodo = per.periodo AND si.product_id = pf.product_id
              )
          AND per.periodo BETWEEN pf.periodo_min AND pf.periodo_max
        ORDER BY 1, 2
    """)

# actualizo precios cuidados en las filas de cero
con.execute("""
    UPDATE tb_zeroes z
    SET plan_precios_cuidados = 1
    WHERE EXISTS (SELECT 1 FROM tb_precios_cuidados p
                  WHERE z.periodo BETWEEN p.periodo_min AND p.periodo_max
                    AND p.product_id = z.product_id)
""")

n_zeroes = con.sql("SELECT COUNT(*) FROM tb_zeroes").fetchone()[0]
print(f"filas de cero agregadas: {n_zeroes:,}")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()

con.execute("""
    CREATE OR REPLACE TABLE tb_sellin_zeroes AS
    SELECT * FROM tb_sellin
    UNION ALL
    SELECT * FROM tb_zeroes
    ORDER BY 1, 2, 3
""")

n_real  = con.sql("SELECT COUNT(*) FROM tb_sellin").fetchone()[0]
n_total = con.sql("SELECT COUNT(*) FROM tb_sellin_zeroes").fetchone()[0]
print(f"filas reales     : {n_real:,}")
print(f"filas totales    : {n_total:,}")
print(f"filas de cero    : {n_total - n_real:,}  ({100*(n_total - n_real)/n_total:.0f}% del dataset)")

# chequeo: ninguna fila real se perdio ni se duplico en el UNION
dup = con.sql("""
    SELECT COUNT(*) FROM (
        SELECT customer_id, product_id, periodo, COUNT(*) AS n
        FROM tb_sellin_zeroes
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
assert dup == 0, f"hay {dup} combinaciones (cliente, producto, periodo) duplicadas tras el UNION"
print("chequeo: sin duplicados (cliente, producto, periodo) tras el UNION -> ok")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()

path_out = DIR_PRE / NOMBRE
con.execute(f"""
    COPY (SELECT * FROM tb_sellin_zeroes ORDER BY 1, 2, 3)
    TO '{path_out}' (FORMAT parquet)
""")

with open(DIR_PRE / NOMBRE.replace(".parquet", "_param.json"), "w", encoding="utf-8") as f:
    json.dump({"nombre": NOMBRE, "param": PARAM,
               "filas_reales": n_real, "filas_totales": n_total},
              f, indent=2, ensure_ascii=False)

print(f"Guardado: {path_out}")
print(f"  {n_total:,} filas")
print(f"  tamanio en disco: {path_out.stat().st_size / 1e6:.0f} MB")
print(f"[{time.time()-t0:.0f}s]")
